# 01 · Train the distraction models
Trains three models on the **driver-wise** split (validation = drivers never seen in training):

| Model | Role |
|---|---|
| `baseline` | small CNN from scratch — the reference number |
| `efficientnet_b0` | main model (ImageNet-pretrained, 2-phase training) |
| `mobilenet_v3_large` | lightweight model for the laptop demo |

**Runtime:** Runtime → Change runtime type → **T4 GPU**.

**If Colab disconnects:** reconnect, run sections 1–4 again, then re-run the cell of the model that was training — it **resumes** from the last saved epoch on Drive.

## 1 · Settings

In [ ]:
GITHUB_USER = "YOUR_GITHUB_USERNAME"   # <-- change this
REPO_NAME = "driver-monitoring-system"

## 2 · Check the GPU and mount Drive

In [ ]:
import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv
assert torch.cuda.is_available(), "No GPU! Runtime → Change runtime type → T4 GPU, then run again."

from google.colab import drive
drive.mount("/content/drive")

## 3 · Get the latest code

In [ ]:
import os, sys
REPO_DIR = f"/content/{REPO_NAME}"
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

## 4 · Load config, data and the driver-wise split

In [ ]:
import shutil
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config, class_names
from src.data.prepare import extract_dataset
from src.data.split import check_no_leakage, summarize_split
from src.data.dataset import build_dataloaders
from src.training.train import TrainConfig, train_model, plot_history

cfg = load_config()
CLASS_CODES = list(class_names(cfg))
SEED = cfg["seed"]
TCFG = cfg["train"]
DRIVE_ROOT = Path(cfg["paths"]["drive_root"])
CKPT_ROOT = DRIVE_ROOT / "checkpoints"
MLRUNS = DRIVE_ROOT / "mlruns"

LOCAL_DATA = extract_dataset(DRIVE_ROOT / "data" / cfg["data"]["zip_name"], cfg["paths"]["local_data"])
split_df = pd.read_csv(cfg["paths"]["splits_file"])
check_no_leakage(split_df)
display(summarize_split(split_df))

In [ ]:
def run(model_name: str, smoke: bool = False, restart: bool = False) -> pd.DataFrame:
    """Train one model (or resume it). smoke=True: 1% of data, 1+1 epochs."""
    mcfg = TCFG["models"][model_name]
    run_dir = CKPT_ROOT / (model_name + ("_smoke" if smoke else ""))
    if restart and run_dir.exists():
        shutil.rmtree(run_dir)
        print("Deleted previous run:", run_dir)

    loaders = build_dataloaders(
        split_df, LOCAL_DATA, CLASS_CODES,
        image_size=TCFG["image_size"], batch_size=TCFG["batch_size"],
        num_workers=TCFG["num_workers"],
        fraction=TCFG["smoke_fraction"] if smoke else 1.0, seed=SEED,
    )
    print({k: len(v.dataset) for k, v in loaders.items()}, "images per split")

    train_cfg = TrainConfig(
        model_name=model_name,
        num_classes=len(CLASS_CODES),
        pretrained=mcfg["pretrained"],
        head_epochs=min(mcfg["head_epochs"], 1) if smoke else mcfg["head_epochs"],
        finetune_epochs=1 if smoke else mcfg["finetune_epochs"],
        lr_head=mcfg.get("lr_head", 1e-3),
        lr_finetune=mcfg["lr_finetune"],
        weight_decay=TCFG["weight_decay"],
        label_smoothing=TCFG["label_smoothing"],
        early_stopping_patience=TCFG["early_stopping_patience"],
        seed=SEED,
        extra={"batch_size": TCFG["batch_size"], "image_size": TCFG["image_size"], "smoke": smoke},
    )
    history = train_model(train_cfg, loaders, run_dir, tracking_dir=None if smoke else MLRUNS)
    display(history)
    if len(history) > 1:
        plot_history(history, model_name); plt.show()
    return history

## 5 · Smoke test (~2 min)
1% of the data, 1 head epoch + 1 fine-tune epoch. It only checks that everything runs end to end —
**the accuracy here is meaningless.** Fix any error before starting the full runs.

In [ ]:
_ = run("efficientnet_b0", smoke=True, restart=True)

## 6 · Baseline CNN (~15–25 min)
Trained from scratch. Expect a much lower score than the pretrained models — that's the point of a baseline.

In [ ]:
hist_baseline = run("baseline")

## 7 · EfficientNet-B0 — main model (~20–30 min)
Phase 1: backbone frozen, head only (2 epochs) → Phase 2: full fine-tuning (up to 10 epochs, early stopping).

In [ ]:
hist_effnet = run("efficientnet_b0")

## 8 · MobileNetV3-Large — lightweight model (~15–25 min)

In [ ]:
hist_mobilenet = run("mobilenet_v3_large")

## 9 · Compare (validation set = unseen drivers)
Final, untouched **test-set** numbers come in `02_evaluate.ipynb`.

In [ ]:
from src.training.checkpoint import load_checkpoint

rows = []
for name in ["baseline", "efficientnet_b0", "mobilenet_v3_large"]:
    best = load_checkpoint(CKPT_ROOT / name / "best.pt")
    if best is None:
        rows.append({"model": name, "status": "not trained yet"})
        continue
    params = sum(t.numel() for t in best["model"].values() if t.dtype.is_floating_point)
    rows.append({"model": name, "best_epoch": best["epoch"], **best["val"],
                 "params_M": round(params / 1e6, 2),
                 "size_MB": round((CKPT_ROOT / name / "best.pt").stat().st_size / 1e6, 1)})
pd.DataFrame(rows)

---
✅ **Done.** Best weights are on Drive at `checkpoints/<model>/best.pt`, history in `history.csv`, all runs in MLflow (`mlruns/`).
**Next:** `02_evaluate.ipynb` — test-set metrics, confusion matrix, calibration and Grad-CAM.